In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    BaggingClassifier,
    VotingClassifier,
    StackingClassifier
)

In [3]:
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [4]:
data = pd.read_csv("diabetes.csv")

In [5]:
print("First 5 rows of dataset:")
print(data.head())

First 5 rows of dataset:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148           72.0             35    169.5  33.6   
1            1       85           66.0             29    102.5  26.6   
2            8      183           64.0             32    169.5  23.3   
3            1       89           66.0             23     94.0  28.1   
4            0      137           40.0             35    168.0  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [6]:
print("\nDataset Shape:")
print(data.shape)


Dataset Shape:
(768, 9)


In [7]:
X = data.drop("Outcome", axis=1)
y = data["Outcome"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
model1 = DecisionTreeClassifier(
    random_state=42
)

model2 = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model3 = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)

In [11]:
models = {
    "Decision Tree": model1,
    "Random Forest": model2,
    "AdaBoost": model3
}

print("\n========== INDIVIDUAL MODELS ==========")

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("\n", name)
    print("----------------------------")
    print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(
        precision_score(y_test, y_pred, zero_division=0), 4
    ))
    print("Recall   :", round(
        recall_score(y_test, y_pred, zero_division=0), 4
    ))
    print("F1 Score :", round(
        f1_score(y_test, y_pred, zero_division=0), 4
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


========== INDIVIDUAL MODELS ==========

 Decision Tree
----------------------------
Accuracy : 0.8117
Precision: 0.7451
Recall   : 0.7037
F1 Score : 0.7238
Confusion Matrix:
[[87 13]
 [16 38]]

 Random Forest
----------------------------
Accuracy : 0.8636
Precision: 0.8113
Recall   : 0.7963
F1 Score : 0.8037
Confusion Matrix:
[[90 10]
 [11 43]]

 AdaBoost
----------------------------
Accuracy : 0.8442
Precision: 0.8
Recall   : 0.7407
F1 Score : 0.7692
Confusion Matrix:
[[90 10]
 [14 40]]


In [12]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=50,
    random_state=42
)

bagging.fit(X_train, y_train)

bagging_pred = bagging.predict(X_test)

In [13]:
boosting = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=1,
        random_state=42
    ),
    n_estimators=100,
    random_state=42
)

boosting.fit(X_train, y_train)

boosting_pred = boosting.predict(X_test)

In [14]:
voting = VotingClassifier(
    estimators=[
        ("Decision Tree", model1),
        ("Random Forest", model2),
        ("AdaBoost", model3)
    ],
    voting="hard"
)

voting.fit(X_train, y_train)

voting_pred = voting.predict(X_test)

In [15]:
stacking = StackingClassifier(
    estimators=[
        ("Decision Tree", model1),
        ("Random Forest", model2),
        ("AdaBoost", model3)
    ],
    final_estimator=LogisticRegression(max_iter=1000)
)

stacking.fit(X_train, y_train)

stacking_pred = stacking.predict(X_test)

In [17]:
def display_metrics(name, y_test, y_pred):

    print("\n======================================")
    print(name)
    print("======================================")

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test, y_pred, zero_division=0
    )
    recall = recall_score(
        y_test, y_pred, zero_division=0
    )
    f1 = f1_score(
        y_test, y_pred, zero_division=0
    )

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

In [18]:
display_metrics(
    "BAGGING",
    y_test,
    bagging_pred
)

display_metrics(
    "BOOSTING (AdaBoost)",
    y_test,
    boosting_pred
)

display_metrics(
    "VOTING",
    y_test,
    voting_pred
)

display_metrics(
    "STACKING",
    y_test,
    stacking_pred
)


BAGGING
Accuracy : 0.8831
Precision: 0.8333
Recall   : 0.8333
F1 Score : 0.8333

Confusion Matrix:
[[91  9]
 [ 9 45]]

BOOSTING (AdaBoost)
Accuracy : 0.8442
Precision: 0.8
Recall   : 0.7407
F1 Score : 0.7692

Confusion Matrix:
[[90 10]
 [14 40]]

VOTING
Accuracy : 0.8571
Precision: 0.7963
Recall   : 0.7963
F1 Score : 0.7963

Confusion Matrix:
[[89 11]
 [11 43]]

STACKING
Accuracy : 0.8571
Precision: 0.7963
Recall   : 0.7963
F1 Score : 0.7963

Confusion Matrix:
[[89 11]
 [11 43]]
